# - Tarefa

### Lendo o arquivo json

In [0]:
from pyspark.sql import SparkSession
tbl_custos_importacao = spark.read.option("multiline", "true").json("/Volumes/lh_nautical/datalake/datas/custos_importacao.json")
tbl_custos_importacao.display()

###  Separando as colunas

In [0]:

from pyspark.sql.functions import explode,col
import pandas as pd
#Separando a coluna historic_data em duas colunas para facilitar a manipulação
tbl_custos_importacao = tbl_custos_importacao.select('*',explode(col('historic_data'))).select('*',col('col.*'))


### Modificando o tipo das colunas

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import when, col, regexp_replace
import pyspark.sql.functions as F

tbl_custos_importacao = tbl_custos_importacao.withColumn("start_date", regexp_replace(col("start_date"), "/", "-"))

# Alterando o tipo de dados da coluna start_date
tbl_custos_importacao = tbl_custos_importacao.withColumn("start_date", F.date_format(F.to_date(col("start_date"), "dd-MM-yyyy"), "yyyy-MM-dd").cast("date"))

# Modificando os tipos de dados
tbl_custos_importacao.withColumn('product_id', col('product_id').cast(IntegerType())).withColumn('usd_price', col('usd_price').cast(FloatType()))
tbl_custos_importacao.display()

### Salvando o arquivo

In [0]:
#excluindo as colunas que não serão utilizadas
tbl_custos_importacao= tbl_custos_importacao.drop('col').drop('historic_data')

tbl_custos_importacao.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/lh_nautical/datalake/datas/custos_importacao")

tbl_custos_importacao.display()